In [1]:
import ast
import os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
from tqdm import tqdm
import multiprocessing
import warnings
import time

warnings.filterwarnings('ignore')

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
os.system(f'emapper.py -h')

usage: emapper.py [-h] [-v] [--list_taxa] [--cpu NUM_CPU]
                  [--mp_start_method {fork,spawn,forkserver}] [--resume]
                  [--override] [-i FASTA_FILE]
                  [--itype {CDS,proteins,genome,metagenome}] [--translate]
                  [--annotate_hits_table SEED_ORTHOLOGS_FILE] [-c FILE]
                  [--data_dir DIR] [--genepred {search,prodigal}]
                  [--trans_table TRANS_TABLE_CODE] [--training_genome FILE]
                  [--training_file FILE]
                  [--allow_overlaps {none,strand,diff_frame,all}]
                  [--overlap_tol FLOAT]
                  [-m {diamond,mmseqs,hmmer,no_search,cache,novel_fams}]
                  [--pident PIDENT] [--query_cover QUERY_COVER]
                  [--subject_cover SUBJECT_COVER] [--evalue EVALUE]
                  [--score SCORE] [--dmnd_algo {auto,0,1,ctg}]
                  [--dmnd_db DMND_DB_FILE]
                  [--sensmode {default,fast,mid-sensitive,sensitive,more-se

0

In [3]:
def eggnog_map(acc_n, que):
    result_dir = f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/eggnog_results/{acc_n}'
    if os.path.exists(f'{result_dir}/{acc_n}.emapper.annotations'):
        pass
    else:
        os.makedirs(result_dir, exist_ok=True)
        handle = open(f'/data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
        seq_record = SeqIO.parse(handle, 'genbank')
        os.chdir('/active-data/temp/blastp')
        temp_file = open(f'temp_proteins_{acc_n}.fasta', 'w+')
        for record in seq_record:
            for feature in record.features:
                if feature.type == 'CDS' and 'translation' in feature.qualifiers:
                    fea_rec = SeqRecord(Seq(feature.qualifiers['translation'][0]), 
                                    id='-'.join([acc_n, record.id, feature.qualifiers['locus_tag'][0]]), 
                                    description=feature.qualifiers['product'][0],)
                    SeqIO.write(fea_rec, temp_file, "fasta")
        temp_file.close()
        os.system(f'emapper.py -m diamond -i temp_proteins_{acc_n}.fasta -o {result_dir}/{acc_n} --tax_scope Bacteria --override --cpu 8 >/dev/null 2>&1')
        os.system(f'rm temp_proteins_{acc_n}.fasta')
    que.put(1)

def eggnog_anno(genus_name, acc_list):
    manager = multiprocessing.Manager()
    que = manager.Queue()
    par = 10
    tot = len(acc_list)
    pool = multiprocessing.Pool(par)
    
    for acc_n in acc_list:
        pool.apply_async(eggnog_map, (acc_n, que))
    
    pool.close()
    
    with tqdm(total = len(acc_list), desc=f'{genus_name}({len(acc_list)})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        count = 0
        while True:
            time.sleep(0.01)
            if not que.empty():
                temp_data = que.get(True)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
    
    pool.join()

In [4]:
for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    acc_list = org_data_n['accession'].tolist()
    eggnog_anno(genus_name, acc_list)

Klebsiella(3554): 100%|███████████████████████████████████████| 3.55k/3.55k [6:08:05<00:00, 6.21s/B]
Staphylococcus(2423): 100%|██████████████████████████████████| 2.42k/2.42k [11:43:02<00:00, 17.4s/B]
Pseudomonas(2343): 100%|█████████████████████████████████████| 2.34k/2.34k [30:45:06<00:00, 47.2s/B]
Bacillus(1976): 100%|████████████████████████████████████████| 1.98k/1.98k [17:37:33<00:00, 32.1s/B]
Salmonella(1853): 100%|███████████████████████████████████████| 1.85k/1.85k [9:02:17<00:00, 17.6s/B]
Streptococcus(1599): 100%|███████████████████████████████████| 1.60k/1.60k [17:22:53<00:00, 39.1s/B]
Streptomyces(1359): 100%|████████████████████████████████████| 1.36k/1.36k [13:15:02<00:00, 35.1s/B]
Acinetobacter(1234): 100%|████████████████████████████████████| 1.23k/1.23k [4:27:00<00:00, 13.0s/B]
Helicobacter(416): 100%|██████████████████████████████████████████| 416/416 [4:18:45<00:00, 37.3s/B]
